# Dataset audit: is the Kaggle knee dataset genuinely new data?

This notebook **only reads** images. It answers:

1. How many Kaggle images are **byte-identical** copies (same SHA-256 hash)?
2. How many are **disguised copies**: the same X-ray re-saved, resized, flipped or rotated?
3. How many overlap with the **Mendeley** dataset we already use?
4. How many **genuinely new** images are left per class, and do any copies carry **conflicting labels**?

**How to run:** Runtime → Change runtime type → T4 GPU (optional here) → Save, then **Runtime → Run all**.
Upload the Kaggle zip from your computer when cell 2 asks. No Kaggle account or Google Drive is needed.
Copy the output of the "Summary" cell and send it back.

## 1. Environment check

In [ ]:
import sys, platform
print("Python", sys.version.split()[0], "|", platform.platform())
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU (fine for this notebook)"

## 2. Upload the Kaggle zip from your computer

Click **Choose Files** below and pick `archive (1).zip` from your D: drive (about 1.3 GB).
Keep this browser tab open until the upload reaches 100% (it can take 10-30 minutes).
*Faster alternative:* drag the zip onto the **Files** panel (folder icon on the left) first. This cell
then finds it and skips the upload.
The upload lives only in this temporary Colab session and is deleted when the session ends.

In [ ]:
from pathlib import Path
from google.colab import files
zips = sorted(Path("/content").glob("*.zip"))
if not zips:
    uploaded = files.upload()                       # opens your computer's file picker
    zips = [Path("/content") / name for name in uploaded]
KAGGLE_ZIP = max(zips, key=lambda z: z.stat().st_size)
print("using:", KAGGLE_ZIP.name, round(KAGGLE_ZIP.stat().st_size / 1e9, 2), "GB")
assert KAGGLE_ZIP.stat().st_size > 500e6, "That zip looks too small - is it the full Kaggle download?"

## 3. Unpack both datasets onto the Colab disk

* Kaggle zip: unpacked, then deleted to save disk space.
* Mendeley: downloaded from its public link, with its published SHA-256 checksum verified.

In [ ]:
import shutil, subprocess, zipfile
data = Path("/content/data"); data.mkdir(exist_ok=True)
if not (data / "OS Collected Data").exists():
    with zipfile.ZipFile(KAGGLE_ZIP) as z:
        z.extractall(data)
    KAGGLE_ZIP.unlink()
if not (data / "Osteoporosis Knee X-ray").exists():
    url = "https://data.mendeley.com/public-files/datasets/fxjm8fb6mw/files/b7a95139-dfef-4db3-b3b9-e0a36315f334/file_downloaded"
    subprocess.run(["curl", "-sL", "-o", str(data / "mendeley.zip"), url], check=True)
    out = subprocess.run(["sha256sum", str(data / "mendeley.zip")], capture_output=True, text=True, check=True).stdout
    assert out.startswith("2d4d3a139f30c6b89eb123791af54db8c78ea143b3cfc98c3431665e54ace8b3"), "Mendeley checksum mismatch"
    with zipfile.ZipFile(data / "mendeley.zip") as z:
        z.extractall(data)
    (data / "mendeley.zip").unlink()
print(sorted(p.name for p in data.iterdir()))
assert (data / "OS Collected Data").exists(), "Expected a folder 'OS Collected Data' inside the Kaggle zip"

## 4. Fingerprint every image

For each image we store:
* `sha256`: identical bytes give an identical hash (exact copies).
* a **128×128 standardised thumbnail**, the same method as `ml/data/manifest.py` in the project,
  for the original and for 5 transformed versions (horizontal flip, vertical flip, 180°, 90°, 270°).
  The dot product of two thumbnails equals their **Pearson correlation**. Different patients reach at
  most ≈0.976 in Mendeley, while re-saved or resized copies score ≈1.000, so ≥ 0.985 counts as "same X-ray".

Each image is shrunk straight away so memory stays small.

In [ ]:
import hashlib, collections
from pathlib import Path
import numpy as np, cv2

SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SOURCES = {
    "kaggle":   (Path("/content/data/OS Collected Data"), lambda p: p.parent.name),
    "mendeley": (Path("/content/data/Osteoporosis Knee X-ray"),
                 lambda p: {"normal": "Normal", "osteopenia": "Osteopenia", "osteoporosis": "Osteoporosis"}.get(p.parent.name.lower())),
}
KINDS = ["orig", "hflip", "vflip", "rot180", "rot90", "rot270"]

def to_gray_u8(img):
    if img.ndim == 3:
        img = cv2.cvtColor(img[..., :3], cv2.COLOR_BGR2GRAY)
    if img.dtype == np.uint16:                          # never assume 8-bit
        lo, hi = int(img.min()), int(img.max())
        img = np.zeros_like(img, np.uint8) if hi == lo else ((img.astype(np.float32) - lo) * 255 / (hi - lo)).astype(np.uint8)
    return img

def thumb(g, size=128):
    h, w = g.shape; side = max(h, w)                    # letterbox: pad to square, then resize
    sq = cv2.copyMakeBorder(g, (side-h)//2, side-h-(side-h)//2, (side-w)//2, side-w-(side-w)//2, cv2.BORDER_CONSTANT, value=0)
    v = cv2.resize(sq, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    v = (v - v.mean()) / (v.std() + 1e-6)
    return v / np.sqrt(v.size)                          # unit length: dot product = correlation

rows, vecs, bad = [], {k: [] for k in KINDS}, []
for src, (root, label_of) in SOURCES.items():
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in SUFFIXES or label_of(p) is None:
            continue
        data = p.read_bytes()
        img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_UNCHANGED)
        if img is None:
            bad.append(str(p)); continue
        g = to_gray_u8(img)
        if max(g.shape) > 512:                          # shrink early: saves memory and time
            s = 512 / max(g.shape)
            g = cv2.resize(g, (round(g.shape[1]*s), round(g.shape[0]*s)), interpolation=cv2.INTER_AREA)
        variants = {"orig": g, "hflip": g[:, ::-1], "vflip": g[::-1], "rot180": g[::-1, ::-1],
                    "rot90": np.rot90(g), "rot270": np.rot90(g, 3)}
        for k in KINDS:
            vecs[k].append(thumb(np.ascontiguousarray(variants[k])))
        rows.append({"source": src, "path": str(p.relative_to(root)), "label": label_of(p),
                     "sha256": hashlib.sha256(data).hexdigest(), "height": img.shape[0], "width": img.shape[1],
                     "dtype": str(img.dtype), "channels": 1 if img.ndim == 2 else img.shape[2]})
vecs = {k: np.stack(v) for k, v in vecs.items()}
print("images:", collections.Counter((r["source"], r["label"]) for r in rows))
print("unreadable:", len(bad), bad[:5])

## 5. Compare every pair, including flipped and rotated versions

In [ ]:
N = len(rows)
best = np.full((N, N), -1.0, np.float32); kind = np.zeros((N, N), np.int8)
for ki, k in enumerate(KINDS):
    C = vecs["orig"] @ vecs[k].T                        # all N x N correlations in one matrix product
    m = C > best; best[m] = C[m]; kind[m] = ki
np.fill_diagonal(best, -1)
sym = np.maximum(best, best.T)
THRESH = 0.985
vals = sym[np.triu_indices(N, 1)]
print("pair correlation quantiles 50/90/99/99.9%:", np.round(np.quantile(vals, [.5, .9, .99, .999]), 3))
print("histogram:", list(zip([0.9, 0.95, 0.97, 0.98, 0.985, 0.99, 0.995, 0.999],
                             np.histogram(vals, [0.9, 0.95, 0.97, 0.98, 0.985, 0.99, 0.995, 0.999, 1.01])[0])))

# Same SHA-256 -> same X-ray too (covers exact copies even if thumbnails differ)
pairs = set(zip(*np.nonzero(np.triu(sym, 1) >= THRESH)))
by_sha = collections.defaultdict(list)
for i, r in enumerate(rows):
    by_sha[r["sha256"]].append(i)
for members in by_sha.values():
    pairs |= {(a, b) for a in members for b in members if a < b}

parent = list(range(N))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in pairs:
    parent[find(a)] = find(b)
for i, r in enumerate(rows):
    r["group"] = int(find(i))
    r["max_corr"] = float(sym[i].max())
    r["best_match"] = rows[int(sym[i].argmax())]["source"] + ":" + rows[int(sym[i].argmax())]["path"]
transforms = collections.Counter(KINDS[kind[a, b] if best[a, b] >= best[b, a] else kind[b, a]] for a, b in pairs if sym[a, b] >= THRESH)
print("matched pairs by transform:", dict(transforms))

## 6. Summary (copy this output and send it back)

In [ ]:
groups = collections.defaultdict(list)
for r in rows:
    groups[r["group"]].append(r)
k_rows = [r for r in rows if r["source"] == "kaggle"]
exact = collections.Counter(r["sha256"] for r in k_rows)
print("KAGGLE FILES:", len(k_rows), dict(collections.Counter(r["label"] for r in k_rows)))
print("  formats:", dict(collections.Counter(Path(r["path"]).suffix.lower() for r in k_rows)),
      "| dtypes:", dict(collections.Counter(r["dtype"] for r in k_rows)),
      "| channels:", dict(collections.Counter(r["channels"] for r in k_rows)))
print("  most common sizes:", collections.Counter((r["height"], r["width"]) for r in k_rows).most_common(4))
print("  byte-identical duplicates inside Kaggle:", sum(c - 1 for c in exact.values() if c > 1))
m_sha = {r["sha256"] for r in rows if r["source"] == "mendeley"}
print("  Kaggle files byte-identical to a Mendeley file:", sum(r["sha256"] in m_sha for r in k_rows))

summary = collections.Counter(); new_per_class = collections.Counter(); copies = []
for members in groups.values():
    ks = [r for r in members if r["source"] == "kaggle"]
    if not ks:
        continue
    labels = {r["label"] for r in members}
    with_mendeley = any(r["source"] == "mendeley" for r in members)
    copies.append(len(ks))
    if len(labels) > 1:
        summary["groups with CONFLICTING labels"] += 1
        summary["kaggle files in conflicting groups"] += len(ks)
    elif with_mendeley:
        summary["groups that are Mendeley images"] += 1
        summary["kaggle files that are Mendeley copies"] += len(ks)
    else:
        summary["genuinely new groups"] += 1
        new_per_class[next(iter(labels))] += 1
print("GROUPS:", dict(summary))
print("GENUINELY NEW X-RAYS (one per group):", dict(new_per_class), "total", sum(new_per_class.values()))
print("Kaggle files per group (copies):", sorted(collections.Counter(copies).items())[:10])

## 7. Download the audit table

This downloads `audit_groups.csv` (one row per image: source, path, label, hash, group, best match) to your
computer. Put it in `D:/Projects/Oestoporosis/New folder/data/`; the project uses it to merge the
datasets without leakage.

In [ ]:
import csv
from google.colab import files
out = Path("/content/audit_groups.csv")
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("rows:", len(rows))
files.download(str(out))